In [ ]:
import pandas as pd
import openpyxl
from openpyxl import load_workbook
from openpyxl.styles import Font, Border, Side, Alignment, PatternFill

# 1. pandasによるデータの集計・連結
# ①前準備
# Excelファイルの読み込みとデータフレーム化
df1 = pd.read_excel('2022_年間売上表.xlsx','Sheet1')
df2 = pd.read_excel('2023_年間売上表.xlsx','Sheet1')

# ②集計
# 商品ごとにグループ化し、金額（千円）の合計金額を算出する
grouped1 = df1.groupby(['商品','売上年']).agg({'金額（千円）': 'sum'})
grouped2 = df2.groupby(['商品','売上年']).agg({'金額（千円）': 'sum'})

# ③データの結合
# 空のデータフレームを作成
df_concat = pd.DataFrame()
# 2つのデータフレームを結合させる
df_concat = pd.concat([grouped1,grouped2], axis = 0).reset_index()

# ④データ整理
# データフレームをソートする
df_sort = df_concat.sort_values(['商品','売上年'], ascending=True)
# 列名を「金額（千円）」から「金額(千円)」に変更する(renameはデータフレームの列名を直接変更しない)
df_rename = df_sort.rename(columns={'金額（千円）': '金額(千円)'})

# ⑤Excelファイルを作成
writer = pd.ExcelWriter('売上集計表.xlsx')
df_rename.to_excel(writer,sheet_name='Sheet1', index=False)
writer.close()

# 2. openpyxlによるデザイン設定
# ①Excelファイルの読み込み
wb = openpyxl.load_workbook('売上集計表.xlsx')
ws = wb['Sheet1']

# ②フォント等デザイン設定
# ヘッダーの設定
# 各列の幅を調整
ws.column_dimensions['A'].width =18
ws.column_dimensions['B'].width =8
ws.column_dimensions['C'].width =14
# Side属性で線のスタイルを指定
thick_border = Side(style='thick')
# Border属性で各辺の線のスタイルを指定
border = Border(top=thick_border, bottom=thick_border, left=thick_border, right=thick_border)
# フォントの設定
new_font_header = Font(name='Meiryo UI', size=11,bold=True, color='000000')
patternFill_header = PatternFill(patternType='solid', fgColor='F2F2F2')
Alignment_header = Alignment(horizontal='center', vertical='bottom')
# ヘッダーの列のセルにフォントを設定
for col in ws.iter_cols(max_row=1):
    for cell in col:
        cell.font = new_font_header
        cell.fill = patternFill_header
        cell.alignment = Alignment_header

# カテゴリーの設定
new_font_category = Font(name='Meiryo UI', size=11, color='000000')
Alignment_category = Alignment(horizontal='general', vertical='bottom')
# カテゴリーの各行のセルにフォントを設定
for row in ws.iter_rows(min_row=2, min_col=1, max_col=1):
    for cell in row:
        cell.font = new_font_category
        cell.alignment = Alignment_category

# データの設定
new_font_data = Font(name='Arial', size=11, color='000000')
Alignment_data = Alignment(horizontal='general', vertical='bottom')
# データの各セルにフォントを設定
for row in ws.iter_rows(min_row=2, min_col=2):
    for cell in row:
        cell.font = new_font_data
        cell.alignment = Alignment_data

# ③Excelファイルに上書き
wb.save('売上集計表.xlsx')